# 🌊 Streaming LLM Responses

**Real-time token-by-token output for better UX**

---

## 📋 Overview

**What you'll learn:**
- Why streaming matters
- Implement streaming with OpenAI/Groq
- Build real-time chat interfaces
- Handle streaming errors
- Server-Sent Events (SSE)

**Time estimate:** ⏱️ 40 minutes | **Difficulty:** 🟡 Intermediate

---

## 🎯 Why Streaming?

**Without streaming:**
```
User waits... ⏳
User waits... ⏳  (10 seconds)
Full response appears!
```

**With streaming:**
```
"The" appears
"answer" appears
"is..." appears (feels instant!)
```

**Benefits:**
- ⚡ Perceived speed increase
- 👍 Better user experience
- 🛑 Can stop early if needed
- 📊 Real-time progress

In [ ]:
import os
from dotenv import load_dotenv
from groq import Groq
import time

load_dotenv()
client = Groq(api_key=os.getenv('GROQ_API_KEY'))

print("✅ Setup complete")

## 🔄 Basic Streaming

In [ ]:
# Non-streaming (wait for full response)
print("Non-streaming (waits for all):")
start = time.time()
response = client.chat.completions.create(
    model="mixtral-8x7b-32768",
    messages=[{"role": "user", "content": "Count from 1 to 10"}],
    stream=False
)
print(response.choices[0].message.content)
print(f"Took: {time.time()-start:.2f}s\n")

# Streaming (get tokens as they arrive)
print("Streaming (tokens appear gradually):")
start = time.time()
stream = client.chat.completions.create(
    model="mixtral-8x7b-32768",
    messages=[{"role": "user", "content": "Count from 1 to 10"}],
    stream=True
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end='', flush=True)
        time.sleep(0.05)  # Slow down for visibility

print(f"\nTook: {time.time()-start:.2f}s")

## 🏗️ Production Streaming Class

In [ ]:
from typing import Iterator, Dict, Any

class StreamingLLM:
    """Production-ready streaming LLM wrapper."""
    
    def __init__(self):
        self.client = Groq(api_key=os.getenv('GROQ_API_KEY'))
    
    def stream_completion(self, prompt: str, **kwargs) -> Iterator[Dict[str, Any]]:
        """Stream completion with metadata."""
        stream = self.client.chat.completions.create(
            model="mixtral-8x7b-32768",
            messages=[{"role": "user", "content": prompt}],
            stream=True,
            **kwargs
        )
        
        for chunk in stream:
            if chunk.choices[0].delta.content:
                yield {
                    'content': chunk.choices[0].delta.content,
                    'finish_reason': chunk.choices[0].finish_reason
                }
    
    def collect_stream(self, prompt: str, **kwargs) -> str:
        """Collect full response from stream."""
        full_response = ""
        for chunk in self.stream_completion(prompt, **kwargs):
            full_response += chunk['content']
        return full_response

# Test
llm = StreamingLLM()

print("Streaming response:")
for chunk in llm.stream_completion("Explain streaming in one sentence"):
    print(chunk['content'], end='', flush=True)
print()

## ✅ Summary

### Key Points:
- 🌊 **Streaming**: Better UX, feels faster
- 🔄 **Implementation**: `stream=True` parameter
- 📦 **Chunks**: Process token-by-token
- 🛡️ **Error handling**: Wrap in try-except

### When to Use:
- ✅ Chat interfaces
- ✅ Long-form content
- ✅ Real-time applications
- ❌ Short responses (overhead not worth it)

### Next: `02_llm_basics/04_cost_calculation.ipynb`